In [ ]:
!unzip -q dataset.zip -d dataset/

!mkdir -p data/

!cp dataset/tables.json data/tables.json


replace dataset/dev.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
!pip install -q -U pyarrow transformers datasets peft trl bitsandbytes accelerate
print("Библиотеки установлены. ТЕПЕРЬ ОБЯЗАТЕЛЬНО ПЕРЕЗАПУСТИТЕ СЕАНС (Runtime -> Restart session)!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 134.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 52.9 MB/s eta 0:00:00
Библиотеки установлены. ТЕПЕРЬ ОБЯЗАТЕЛЬНО ПЕРЕЗАПУСТИТЕ СЕАНС (Runtime -> Restart session)!


In [ ]:
mport json

with open('dataset/train_spider.json', 'r', encoding='utf-8') as f:
    spider_data = json.load(f)

with open('data/train_instructions.jsonl', 'w', encoding='utf-8') as out_f:
    for item in spider_data:
        instruction_item = {
            "instruction": "Преобразуй вопрос на естественном языке в SQL-запрос к базе данных.",
            "input": item["question"],
            "output": item["query"]
        }
        out_f.write(json.dumps(instruction_item, ensure_ascii=False) + '\n')

print("Обучающий датасет успешно конвертирован и сохранен в data/!")

Обучающий датасет успешно конвертирован и сохранен в data/!


In [ ]:
import json

with open('dataset/dev.json', 'r', encoding='utf-8') as f:
    spider_data = json.load(f)

with open('data/val_instructions.jsonl', 'w', encoding='utf-8') as out_f:
    for item in spider_data:
        instruction_item = {
            "instruction": "Преобразуй вопрос на естественном языке в SQL-запрос к базе данных.",
            "input": item["question"],
            "output": item["query"]
        }
        out_f.write(json.dumps(instruction_item, ensure_ascii=False) + '\n')

print("Валидационный датасет успешно конвертирован и сохранен в data/!")

Валидационный датасет успешно конвертирован и сохранен в data/!


In [ ]:
import pandas as pd
import json

print("--- Примеры из обучающей выборки (data/train_instructions.jsonl) ---")
train_df = pd.read_json('data/train_instructions.jsonl', lines=True)
display(train_df.head(3))

print("\n--- Примеры из валидационной выборки (data/val_instructions.jsonl) ---")
val_df = pd.read_json('data/val_instructions.jsonl', lines=True)
display(val_df.head(3))

print("\n--- Структура баз данных (data/tables.json) ---")
with open('data/tables.json', 'r', encoding='utf-8') as f:
    tables_data = json.load(f)

first_db = tables_data[0]
print(f"ID базы данных: {first_db.get('db_id')}")
print(f"Таблицы в этой БД: {first_db.get('table_names')}")
print("Колонки (индекс таблицы, название):")
for col in first_db.get('column_names')[:5]:
    print(col)

--- Примеры из обучающей выборки (data/train_instructions.jsonl) ---


,instruction,input,output
0,Преобразуй вопрос на естественном языке в SQL-...,How many heads of the departments are older th...,SELECT count(*) FROM head WHERE age > 56
1,Преобразуй вопрос на естественном языке в SQL-...,"List the name, born state and age of the heads...","SELECT name , born_state , age FROM head ORD..."
2,Преобразуй вопрос на естественном языке в SQL-...,"List the creation year, name and budget of eac...","SELECT creation , name , budget_in_billions ..."



--- Примеры из валидационной выборки (data/val_instructions.jsonl) ---


,instruction,input,output
0,Преобразуй вопрос на естественном языке в SQL-...,How many singers do we have?,SELECT count(*) FROM singer
1,Преобразуй вопрос на естественном языке в SQL-...,What is the total number of singers?,SELECT count(*) FROM singer
2,Преобразуй вопрос на естественном языке в SQL-...,"Show name, country, age for all singers ordere...","SELECT name , country , age FROM singer ORDE..."



--- Структура баз данных (data/tables.json) ---
ID базы данных: perpetrator
Таблицы в этой БД: ['perpetrator', 'people']
Колонки (индекс таблицы, название):
[-1, '*']
[0, 'perpetrator id']
[0, 'people id']
[0, 'date']
[0, 'year']


In [ ]:
import json

def show_data_format(filepath='data/train_instructions.jsonl'):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            first_line = f.readline()

        data_sample = json.loads(first_line)

        print("Формат записи: JSON-объект (одна строка из файла .jsonl)\n")
        print("Структура полей:")

        for key, value in data_sample.items():
            print(f"Поле '{key}':")
            print(f"  Содержимое: {value}")
            print("-" * 50)


    except FileNotFoundError:
        print(f"Ошибка: Файл {filepath} не найден.")

show_data_format()

Формат записи: JSON-объект (одна строка из файла .jsonl)

Структура полей:
Поле 'instruction':
  Содержимое: Преобразуй вопрос на естественном языке в SQL-запрос к базе данных.
--------------------------------------------------
Поле 'input':
  Содержимое: How many heads of the departments are older than 56 ?
--------------------------------------------------
Поле 'output':
  Содержимое: SELECT count(*) FROM head WHERE age  >  56
--------------------------------------------------


In [ ]:
import pandas as pd
import os
from datetime import datetime

def generate_dataset_card(filepath='data/train_instructions.jsonl', output_path='data/README.md'):
    print("--- Генерация метаданных (Раздел 4.3) ---\n")

    if not os.path.exists(filepath):
        print(f"Ошибка: Файл {filepath} не найден. Убедитесь, что папка data/ создана.")
        return

    df = pd.read_json(filepath, lines=True)

    avg_input_words = df['input'].apply(lambda x: len(str(x).split())).mean()
    avg_output_words = df['output'].apply(lambda x: len(str(x).split())).mean()
    total_rows = len(df)

    current_date = datetime.now().strftime("%Y-%m")

    readme_content = f"""# Dataset Card: Text-to-SQL (Theme 50)

## Общая информация
* **Тип задачи:** Instruction Tuning (Преобразование естественного языка в SQL)
* **Источник текстов:** Открытый датасет Spider (адаптированный)
* **Дата формирования:** {current_date}
* **Язык текста:** Многоязычный (Инструкции - RU, Данные - EN)

## Статистика набора данных (train)
* **Общее количество примеров:** {total_rows}
* **Средняя длина пользовательского запроса (input):** {avg_input_words:.1f} слов
* **Средняя длина генерируемого ответа (output):** {avg_output_words:.1f} слов

## Структура
Датасет представлен в формате JSONL, содержащем тройки "instruction", "input" и "output". Дополнительно прилагается файл `tables.json` со схемами реляционных баз данных.
"""

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(readme_content)

    print(f"Файл {output_path} успешно сгенерирован!\n")
    print("--- Содержимое README.md ---")
    print(readme_content)

generate_dataset_card()

--- Генерация метаданных (Раздел 4.3) ---

Файл data/README.md успешно сгенерирован!

--- Содержимое README.md ---
# Dataset Card: Text-to-SQL (Theme 50)

## Общая информация
* **Тип задачи:** Instruction Tuning (Преобразование естественного языка в SQL)
* **Источник текстов:** Открытый датасет Spider (адаптированный)
* **Дата формирования:** 2026-05
* **Язык текста:** Многоязычный (Инструкции - RU, Данные - EN)

## Статистика набора данных (train)
* **Общее количество примеров:** 7000
* **Средняя длина пользовательского запроса (input):** 12.7 слов
* **Средняя длина генерируемого ответа (output):** 15.9 слов

## Структура
Датасет представлен в формате JSONL, содержащем тройки "instruction", "input" и "output". Дополнительно прилагается файл `tables.json` со схемами реляционных баз данных.



In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": "data/train_instructions.jsonl",
    "val": "data/val_instructions.jsonl"
})

def format_instruction(example):
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Output:\n{example['output']} </s>"}

mapped_dataset = dataset.map(format_instruction)
print("Данные подготовлены для Unsloth.")

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

Данные подготовлены для Unsloth.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Настройка 4-битного квантования (compute_dtype = float32 для стабильности)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32 # ПРИНУДИТЕЛЬНО float32
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float32 # ПРИНУДИТЕЛЬНО float32
)

# Конфигурация LoRA
lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("Модель загружена в режиме полной совместимости (Float32).")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Модель загружена в режиме полной совместимости (Float32).


In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./sql_model_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    optim="paged_adamw_8bit",
    logging_steps=10,
    learning_rate=1e-4,

    fp16=False,
    bf16=False,

    max_steps=100,
    save_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    dataset_text_field="text",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=mapped_dataset['train'],
    eval_dataset=mapped_dataset['val'],
    args=training_args,
)

model.config.use_cache = False

print("Запуск обучения в режиме максимальной совместимости (без AMP)...")
trainer.train()

trainer.model.save_pretrained("text-to-sql-lora-adapter")
print("Обучение завершено. Адаптер успешно сохранен!")

Adding EOS to train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1034 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1034 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Запуск обучения в режиме максимальной совместимости (без AMP)...


Step,Training Loss,Validation Loss
50,0.862221,0.843655
100,0.752950,0.760432


Обучение завершено. Адаптер успешно сохранен!


In [ ]:
import os
import zipfile
from google.colab import files

def create_archive(archive_name, folders_to_include):
    print(f"Начинаю упаковку файлов в {archive_name}...")

    with zipfile.ZipFile(archive_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for folder in folders_to_include:
            if os.path.exists(folder):
                for root, dirs, files_in_folder in os.walk(folder):
                    for file in files_in_folder:
                        file_path = os.path.join(root, file)
                        zipf.write(file_path, file_path)
                print(f"[OK] Папка '{folder}' добавлена.")
            else:
                print(f"[!] Предупреждение: Папка '{folder}' не найдена и была пропущена.")

    print(f"\nАрхив {archive_name} успешно создан.")

    print("Запрос на скачивание отправлен...")
    files.download(archive_name)

folders = ['data', 'text-to-sql-lora-adapter']
archive = 'course_project_results.zip'

create_archive(archive, folders)

Начинаю упаковку файлов в course_project_results.zip...
[OK] Папка 'data' добавлена.
[OK] Папка 'text-to-sql-lora-adapter' добавлена.

Архив course_project_results.zip успешно создан.
Запрос на скачивание отправлен...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = "text-to-sql-lora-adapter"
val_file = "data/val_instructions.jsonl"

print("Загрузка модели и адаптера (это может занять минуту)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map={"": 0}
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

print("-" * 50)
print("СИСТЕМА ГОТОВА К РАБОТЕ")
print("-" * 50)

with open(val_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 10: break

        data = json.loads(line)
        question = data['input']

        prompt = f"### Instruction:\nПреобразуй вопрос на естественном языке в SQL-запрос к базе данных.\n\n### Input:\n{question}\n\n### Output:\n"

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=64,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        sql_answer = full_text.split("### Output:")[1].strip()

        print(f"ПРИМЕР №{i+1}")
        print(f"ВОПРОС: {question}")
        print(f"SQL-ОТВЕТ: {sql_answer}")
        print("-" * 30)

print("\nТестирование завершено!")

Загрузка модели и адаптера (это может занять минуту)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

--------------------------------------------------
СИСТЕМА ГОТОВА К РАБОТЕ
--------------------------------------------------


[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ПРИМЕР №1
ВОПРОС: How many singers do we have?
SQL-ОТВЕТ: SELECT count(*) FROM singers
------------------------------


...

ПРИМЕР №7
ВОПРОС: Show the name and the release year of the song by the youngest singer.
SQL-ОТВЕТ: SELECT name ,  release_year FROM song WHERE singer_id  IS  min(singer_id)  ORDER BY release_year  LIMIT 1
------------------------------


[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ПРИМЕР №8
ВОПРОС: What are the names and release years for all the songs of the youngest singer?
SQL-ОТВЕТ: SELECT T1.name ,  T2.release_year FROM song AS T1 JOIN singer AS T2 ON T1.singer_id  =  T2.singer_id WHERE T1.singer_id  =  T2.youngest_singer_id
------------------------------


[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ПРИМЕР №9
ВОПРОС: What are all distinct countries where singers above age 20 are from?
SQL-ОТВЕТ: SELECT DISTINCT country FROM singers WHERE age >= 20
------------------------------
ПРИМЕР №10
ВОПРОС: What are  the different countries with singers above age 20?
SQL-ОТВЕТ: SELECT DISTINCT country FROM singer WHERE age >= 20
------------------------------

Тестирование завершено!
